# Text Generation - QA
https://youtu.be/WZeZZ8_W-M4?si=b_ZD7Pvl2zTexE_x&t=12761


## Setup

In [ ]:
%pip install boto3

In [ ]:
import json
import os
import sys
import boto3

client = boto3.client("bedrock-runtime")

# Example 1

## Data

In [ ]:
prompt_data = """You are an helpful assistant. Answer questions in a concise way. If you are unsure about the
answer say 'I am unsure'

Question: How can I fix a flat tire on my Audi A8?
Answer:"""

## Call API

In [ ]:
params = {
    "maxTokenCount":512,
    "stopSequences":[],
    "temperature":0,
    "topP":0.9
}

In [ ]:
body=json.dumps({
        "inputText": prompt_data,
        "textGenerationConfig": params,
    })

In [ ]:
response = client.invoke_model(
    modelId="amazon.titan-tg1-large", 
    accept='application/json', 
    contentType='application/json'
    body=body, 
)

In [ ]:
body = json.loads(response.get('body').read())

## Display response

In [ ]:
print(body.get('results')[0].get('outputText'))

***
# Example 2 (hallucination)

We provide a fictional car model.

## Data

In [ ]:
prompt_data = "How can I fix a flat tire on my Amazon Tirana?"

## Call API
Using new prompt data and same params.

In [ ]:
body=json.dumps({
        "inputText": prompt_data,
        "textGenerationConfig": params,
    })

In [ ]:
response = client.invoke_model(
    modelId="amazon.titan-tg1-large", 
    accept='application/json', 
    contentType='application/json'
    body=body, 
)

In [ ]:
body = json.loads(response.get('body').read())

## Display response

In [ ]:
print(body.get('results')[0].get('outputText'))

***
# Example 3 (RAG)

Fix hallucination by providing some context to the model to augment knowledge base on the fly.

## Data

In [ ]:
context = """Tires and tire pressure for Audi A8:

Tires are made of black rubber and are mounted on the wheels of your vehicle. They provide the necessary grip for driving, cornering, and braking. Two important factors to consider are tire pressure and tire wear, as they can affect the performance and handling of your car.

Where to find recommended tire pressure:

You can find the recommended tire pressure specifications on the inflation label located on the driver's side B-pillar of your vehicle. Alternatively, you can refer to your vehicle's manual for this information. The recommended tire pressure may vary depending on the speed and the number of occupants or maximum load in the vehicle.

Reinflating the tires:

When checking tire pressure, it is important to do so when the tires are cold. This means allowing the vehicle to sit for at least three hours to ensure the tires are at the same temperature as the ambient temperature.

To reinflate the tires:

    Check the recommended tire pressure for your vehicle.
    Follow the instructions provided on the air pump and inflate the tire(s) to the correct pressure.
    In the center display of your vehicle, open the "Car status" app.
    Navigate to the "Tire pressure" tab.
    Press the "Calibrate pressure" option and confirm the action.
    Drive the car for a few minutes at a speed above 30 km/h to calibrate the tire pressure.

Note: In some cases, it may be necessary to drive for more than 15 minutes to clear any warning symbols or messages related to tire pressure. If the warnings persist, allow the tires to cool down and repeat the above steps.

Flat Tire:

If you encounter a flat tire while driving, you can temporarily seal the puncture and reinflate the tire using a tire mobility kit. This kit is typically stored under the lining of the luggage area in your vehicle.

Instructions for using the tire mobility kit:

    Open the tailgate or trunk of your vehicle.
    Lift up the lining of the luggage area to access the tire mobility kit.
    Follow the instructions provided with the tire mobility kit to seal the puncture in the tire.
    After using the kit, make sure to securely put it back in its original location.
    Contact Audi or an appropriate service for assistance with disposing of and replacing the used sealant bottle.

Please note that the tire mobility kit is a temporary solution and is designed to allow you to drive for a maximum of 10 minutes or 8 km (whichever comes first) at a maximum speed of 80 km/h. It is advisable to replace the punctured tire or have it repaired by a professional as soon as possible."""

In [ ]:
question = "How can I fix a flat tire on my Audi A8?"

In [ ]:
prompt_data = f"""Answer the question based only on the information provided between ## and give step by step guide.
Only answer the question if the for the exact made and model of the car, otherwise, answer "I don't know".
#
{context}
#

Question: {question}
Answer:"""

## Call API

In [ ]:
body = json.dumps({
        "inputText": prompt_data,
        "textGenerationConfig": params,
    })

In [ ]:
resp = client.invoke_model(
    modelId="amazon.titan-tg1-large", 
    accept='application/json', 
    contentType='application/json'
    body=body, 
)

In [ ]:
body = json.loads(resp.get('body').read())

## Display response

In [ ]:
answer = body.get('results')[0].get('outputText')
print(answer)

***
# Example 4 (Bedrock Streaming)

Generates output as model is generating tokens.

In [ ]:
from IPython.display import display_markdown,Markdown,clear_output

resp = client.invoke_model_with_response_stream(
    modelId="amazon.titan-tg1-large", 
    accept='application/json', 
    contentType='application/json'
    body=body, 
)

In [ ]:
stream = resp.get('body')
output = []
i = 1

if stream:
    for event in stream:
        chunk = event.get('chunk')
        if chunk:
            chunk_obj = json.loads(chunk.get('bytes').decode())
            text = chunk_obj['outputText']
            clear_output(wait=True)
            output.append(text)
            display_markdown(Markdown(''.join(output)))
            i+=1